## LAST_FM

In [3]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time
import requests, time, re, html
from io import BytesIO
from dotenv import load_dotenv
import os


In [ ]:
load_dotenv()
my_API_KEY = os.getenv("LASTFM_API_KEY")
SHARED_SECRET = os.getenv("LASTFM_SHARED_SECRET")

In [4]:
def get_artist_info(artist_name): # Generamos una función para obtener de la API la información de los artistas seleccionados en la extracción de la API de Spotipy
    API_KEY = my_API_KEY 
    BASE_URL = "https://ws.audioscrobbler.com/2.0/" # Conexión a la API de LastFM

    params = {
        "method": "artist.getinfo",
        "artist": artist_name,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect": 1,
        "lang": "en"
    }                                               # Parametros dados a la API para la extración en el endpoint "artist.getinfo"

    response = requests.get(BASE_URL, params=params)
    data = response.json() 

    # Validar si hay error
    if "error" in data:
        print(f"❌ Error: {data['message']}")
        return None

    artist = data["artist"]
    listeners = artist["stats"]["listeners"]
    playcount = artist["stats"]["playcount"]
    similar_artists = [a["name"] for a in artist["similar"]["artist"]]
    similar_artist = similar_artists[0] if similar_artists else None
     # Información de cada canción según los parametros explicitos

    bio = data.get("artist", {}).get("bio", {})
    summary = bio.get("summary",{})

    return {
        "artist_name": artist["name"],
        "biography": summary,
        "listeners": listeners,
        "playcount": playcount,
        "similar_artist": similar_artist
    } # Formato para mostrar la información



In [44]:
# Incluir aquí el CSV de los datos de Spotify

In [5]:
df_spotify = pd.read_csv("Canciones_Spotify.csv")  # Transformación en DataFrame
df_spotify.head()

,id,track_name,artist_name,year,genre,album_type,release_date,popularity
0,1WHNqqRWhJVZIdCScFKtl5,Washington on Your Side,Leslie Odom Jr.,2015,soundtrack,album,2015-09-25,66
1,1DLfR4MOfLYbV6v3xrmWa8,We Know,Lin-Manuel Miranda,2015,soundtrack,album,2015-09-25,66
2,2AtC6i0b8TjpjhWBZYLprX,Bonetrousle,Toby Fox,2015,soundtrack,album,2015-09-15,58
3,6oF8ueLn5hIl4PRp17sxW6,That Would Be Enough,Phillipa Soo,2015,soundtrack,album,2015-09-25,67
4,46YSff2Rq1ZtN1YVk5cwbZ,Brave Shine,Aimer,2015,soundtrack,album,2015-07-29,61


In [6]:
df_spotify['artist_name'].tolist() # Creación de lista de artistas para ejecutar en archivo "artistas_LastFM.ipynb"

['Leslie Odom Jr.',
 'Lin-Manuel Miranda',
 'Toby Fox',
 'Phillipa Soo',
 'Aimer',
 'Original Broadway Cast of Hamilton',
 'Bear McCreary',
 'Fear, and Loathing in Las Vegas',
 'Okieriete Onaodowan',
 'Toby Fox',
 'Lin-Manuel Miranda',
 'Daveed Diggs',
 'Original Broadway Cast of Hamilton',
 'Leslie Odom Jr.',
 'Daveed Diggs',
 'Christopher Jackson',
 'Original Broadway Cast of Hamilton',
 'Toby Fox',
 'Lin-Manuel Miranda',
 'Jonathan Groff',
 'Leslie Odom Jr.',
 'Toby Fox',
 'Toby Fox',
 'Leslie Odom Jr.',
 'Toby Fox',
 'Lin-Manuel Miranda',
 'Phillipa Soo',
 'Toby Fox',
 'Jasmine Cephas-Jones',
 'Original Broadway Cast of Hamilton',
 'Toby Fox',
 'Leslie Odom Jr.',
 'Leslie Odom Jr.',
 'Thayne Jasperson',
 'Leslie Odom Jr.',
 'KANA-BOON',
 'Michael Giacchino',
 'Leslie Odom Jr.',
 'Jonathan Groff',
 'Lin-Manuel Miranda',
 'Goose house',
 'Renée Elise Goldsberry',
 'John Williams',
 'Toby Fox',
 'Toby Fox',
 'Lin-Manuel Miranda',
 'Phillipa Soo',
 'Bear McCreary',
 'Toby Fox',
 'Lesli

In [7]:
len(df_spotify['artist_name'].tolist()) # Número de registros de la extración de datos de la API Spotipy

3000

In [8]:
unique_artists_spotify = df_spotify['artist_name'].unique().tolist() # Generar valores únicos de artistas de la extración de datos de la API Spotipy

In [9]:
len(unique_artists_spotify) # Número de registros unicos de artistas de la extración de datos de la API Spotipy

1201

In [50]:
lista_artist_lastfm_NN = [] # Variable para almacenar toda la información de todos los artistas que no sean nulos.

for X in unique_artists_spotify:
    info = get_artist_info(X)
    if info is not None:  # solo agrega si tiene datos
        lista_artist_lastfm_NN.append(info)
    time.sleep(0.5)

❌ Error: The artist you supplied could not be found
❌ Error: The artist you supplied could not be found
❌ Error: The artist you supplied could not be found


In [1]:
len(lista_artist_lastfm_NN) # Número de registros unicos de artistas de la extración de datos de la API LastFM

NameError: name 'lista_artist_lastfm_NN' is not defined

In [54]:
df_lastfm = pd.DataFrame(lista_artist_lastfm_NN) # Transformación en DataFrame

In [55]:
df_lastfm

,artist_name,biography,listeners,playcount,similar_artist
0,Leslie Odom Jr.,"Leslie Odom, Jr (born August 6, 1981) is an Am...",783654,33526440,Christopher Jackson
1,Lin-Manuel Miranda,"Lin-Manuel Miranda (born January 16, 1980 in N...",690836,24963003,Leslie Odom Jr.
2,Toby Fox,Toby Fox is an American developer and composer...,1110128,134239421,Lena Raine
3,Phillipa Soo,"Phillipa Soo (born May 31, 1990) is an America...",547924,14230247,Christopher Jackson
4,Aimer,There are at least 4 artists who go by Aimer:\...,446718,23143896,ReoNa
...,...,...,...,...,...
1193,Amaarae,"Ama Serwah Genfi (born July 4, 1994), known pr...",935034,30374468,Rochelle Jordan
1194,Chris Brown,"Christopher Maurice Brown (born May 5, 1989) i...",4598461,146850181,Chris Brown & Tyga
1195,Lola Jane,"<a href=""https://www.last.fm/music/Lola+Jane""...",5731,50670,Archie & Sizzle
1196,Duda,There are two groups using the name Duda one f...,6635,69758,Peled


In [56]:
df_lastfm.to_csv("Artistas_LastFM.csv", index=False) #Extraccion LastFM to CSV

In [11]:
df_artists_lastfm = pd.read_csv("Artistas_LastFM.csv") # Lectura de CSV
df_artists_lastfm

,artist_name,biography,listeners,playcount,similar_artist
0,Leslie Odom Jr.,"Leslie Odom, Jr (born August 6, 1981) is an Am...",783654,33526440,Christopher Jackson
1,Lin-Manuel Miranda,"Lin-Manuel Miranda (born January 16, 1980 in N...",690836,24963003,Leslie Odom Jr.
2,Toby Fox,Toby Fox is an American developer and composer...,1110128,134239421,Lena Raine
3,Phillipa Soo,"Phillipa Soo (born May 31, 1990) is an America...",547924,14230247,Christopher Jackson
4,Aimer,There are at least 4 artists who go by Aimer:\...,446718,23143896,ReoNa
...,...,...,...,...,...
1193,Amaarae,"Ama Serwah Genfi (born July 4, 1994), known pr...",935034,30374468,Rochelle Jordan
1194,Chris Brown,"Christopher Maurice Brown (born May 5, 1989) i...",4598461,146850181,Chris Brown & Tyga
1195,Lola Jane,"<a href=""https://www.last.fm/music/Lola+Jane""...",5731,50670,Archie & Sizzle
1196,Duda,There are two groups using the name Duda one f...,6635,69758,Peled


In [13]:
def clean_html_links(text):
    if not isinstance(text, str):
        return text
    
    # Reemplaza <a href="URL">texto</a> → URL
    text = re.sub(r'<a href="(https?://[^"]+)".*?>.*?</a>', r'\1', text)

    # Quita cualquier otra etiqueta HTML que quede
    text = re.sub(r'<.*?>', '', text)

    return text.strip()


df_artists_lastfm = pd.read_csv("Artistas_LastFM.csv")

for i, artist in enumerate(df_artists_lastfm["artist_name"], start=1):
    biography_text = df_artists_lastfm.loc[i-1, "biography"]     
    cleaned_text = clean_html_links(biography_text)    #aquí limpiamos el codigo HTML
    df_artists_lastfm.loc[i-1, "biography"] = cleaned_text  #guardamos el texto limpio en la misma celda
    print(f"{i}. ✅ {artist}")
    time.sleep(0.25)

1. ✅ Leslie Odom Jr.
2. ✅ Lin-Manuel Miranda
3. ✅ Toby Fox
4. ✅ Phillipa Soo
5. ✅ Aimer
6. ✅ Original Broadway Cast of Hamilton
7. ✅ Bear McCreary
8. ✅ Fear, and Loathing in Las Vegas
9. ✅ Okieriete Onaodowan
10. ✅ Daveed Diggs
11. ✅ Christopher Jackson
12. ✅ Jonathan Groff
13. ✅ Jasmine Cephas-Jones
14. ✅ Thayne Jasperson
15. ✅ KANA-BOON
16. ✅ Michael Giacchino
17. ✅ Goose house
18. ✅ Renée Elise Goldsberry
19. ✅ John Williams
20. ✅ Ariana DeBose
21. ✅ Eir Aoi
22. ✅ Kazuya Yoshii
23. ✅ The Barden Bellas
24. ✅ Ramin Djawadi
25. ✅ Agrupacion Musical Nuestro Padre Jesús Despojado de Jaen
26. ✅ Marcin Przybyłowicz
27. ✅ Trevor Morris
28. ✅ Dario Marianelli
29. ✅ 坂本龍一
30. ✅ Roque Baños
31. ✅ Basil Poledouris
32. ✅ Craig Armstrong
33. ✅ Mikolai Stroinski
34. ✅ BRADIO
35. ✅ Donna Burke
36. ✅ SIE Sound Team
37. ✅ CHiCO with HoneyWorks
38. ✅ Percival Schuttenbach
39. ✅ Ólafur Arnalds
40. ✅ Sho Oosawa
41. ✅ Patrick Doyle
42. ✅ Peter Gregson
43. ✅ Super DJ Yuuki
44. ✅ Yutaka Yamada
45. ✅ Harry G

In [14]:
df_artists_lastfm.tail(10)

,artist_name,biography,listeners,playcount,similar_artist
1188,Praiz,"Praise Ugbede Adejo (born 8 March 1984), bette...",19973,117871,illBliss
1189,Irina Barros,https://www.last.fm/music/Irina+Barros,4786,44513,Rui Orlando
1190,PAAX (Tulum),https://www.last.fm/music/PAAX+(Tulum),39889,272297,Antaares
1191,Matt Sawyer,https://www.last.fm/music/Matt+Sawyer,1755,4391,Stereo Kulisse
1192,Cubita,https://www.last.fm/music/Cubita,5383,50185,Nuno Ribeiro
1193,Amaarae,"Ama Serwah Genfi (born July 4, 1994), known pr...",935034,30374468,Rochelle Jordan
1194,Chris Brown,"Christopher Maurice Brown (born May 5, 1989) i...",4598461,146850181,Chris Brown & Tyga
1195,Lola Jane,https://www.last.fm/music/Lola+Jane,5731,50670,Archie & Sizzle
1196,Duda,There are two groups using the name Duda one f...,6635,69758,Peled
1197,Simone Vitullo,Simone Vitullo DJ and house music producer for...,21699,101880,Chambord


In [15]:
df_artists_lastfm.to_csv("Artistas_LastFM.csv", index=False) #Extraccion LastFM to CSV